# Getting started

In [ ]:
# Standard library imports
import configparser

# Third-party imports
import pandas as pd
import anthropic

# Pandas configuration
pd.set_option('display.max_colwidth', None)

## 1. Data

In [2]:
data_sample = pd.read_csv("../data/twitter_data_clean_sample.csv")

In [3]:
data_sample.head()

,customer_tweet,company_tweet,company
0,Ordered this around 2am Friday morning and it made it here already... good job @115830 https://t.co/XXMuII3QwQ,@383517 I am very happy to hear this Pablo:) I hope you enjoy your order.^GA,AmazonHelp
1,"@AmazonHelp what does ""ships in 1-3 weeks"" actually mean? Do you have the item I want in stock or not? Items like this have given me issues","@274096 If your item will ship in 1-3 weeks, this means the item is not in stock and needs to be ordered from our distributor. More info here: https://t.co/V7JYyWd9JF ^RA",AmazonHelp
2,@115821 // Email from Representative not correct. There was someone to receive package. Whoever said different at @118706 is lying.,@528375 I'm sorry you haven't received your package. We'd like to help. Please contact us here: https://t.co/hApLpMlfHN ^AY,AmazonHelp
3,je l’ai déjà l’application amazon jdevrais être immunisé de vos pubs de merde @115821,"@792999 3/3 Ensuite décochez à nouveau les cases que vous aviez sélectionnées. N'oubliez pas de ""Valider"" pour effectuer vos modifications. \nNous espérons que ces informations vous seront utiles.",AmazonHelp
4,"I must say @115830, a package left under a doormat which is full of holes, in the middle of a downpour, is not the best idea #wetelectronics","@776873 I apologize for how your delivery was handled, that is not the experience we want our customers to have. Which courier was assigned delivery of that package, as shown in the order details here: https://t.co/aaDyEz1VgE ^CH",AmazonHelp


In [4]:
data_sample.company.value_counts()

company
AmazonHelp      100
AppleSupport    100
SpotifyCares    100
Uber_Support    100
Name: count, dtype: int64

## 2. How to use the Anthropic API

### Option A: Using a config.ini file

Create a `config.ini` file containing your Anthropic API credentials:

```
[ANTHROPIC_API]
ANTHROPIC_KEY = your_api_key_here
```

### Option B: Using an environment variable (recommended)

```bash
export ANTHROPIC_API_KEY=your_api_key_here
```

If using the environment variable, the Anthropic SDK will pick it up automatically.

In [5]:
# Option A: Loading API key from configuration file
config = configparser.ConfigParser()
config.read('../config.ini')
ANTHROPIC_KEY = config.get('ANTHROPIC_API', 'ANTHROPIC_KEY')
client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

# Option B: Using environment variable (recommended)
# The SDK automatically reads the ANTHROPIC_API_KEY environment variable
#client = anthropic.Anthropic()

### Messages API : Get Claude's response to your prompt

#### Documentation : https://docs.anthropic.com/en/docs/build-with-claude/overview

In [6]:
message_text = "Placed an order for some headphones early Monday morning, and they've arrived in 2 days... impressive service!"
company = "Amazon"

print(f"Tweet: {message_text} \nCompany: {company}")

Tweet: Placed an order for some headphones early Monday morning, and they've arrived in 2 days... impressive service! 
Company: Amazon


In [7]:
instruction = f"""\
You are a chatbot answering customer's messages. You are working for a company called {company}. Reply to the message below.
#####
Message:
\"{message_text}\" """
print(f"Instruction:\n\n{instruction}")

Instruction:

You are a chatbot answering customer's messages. You are working for a company called Amazon. Reply to the message below.
#####
Message:
"Placed an order for some headphones early Monday morning, and they've arrived in 2 days... impressive service!" 


In [8]:
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": instruction}
    ],
    temperature=0.7  # temperature ranges from 0 (deterministic) to 1 (creative)
)

In [9]:
generated_text_vanilla = response.content[0].text
print(f"Answer generated:\n\n{generated_text_vanilla}")

Answer generated:

Thank you so much for the kind words! We're thrilled to hear that your headphones arrived so quickly and that you're impressed with our service. 🎉

Fast and reliable delivery is something we work hard to provide, so it's wonderful to know we met your expectations. We hope you're enjoying your new headphones!

If you have any questions about your order or need any assistance in the future, please don't hesitate to reach out. We're always here to help.

Thank you for shopping with Amazon! 😊


### Embedding model : Get the embedding of a text

**Note**: Unlike OpenAI, Anthropic does not provide a built-in embedding API. For embeddings, we use open-source models. Here we use `sentence-transformers` which runs locally and is free.

In [10]:
# Install sentence-transformers if not already installed
# !pip install sentence-transformers

from sentence_transformers import SentenceTransformer

# Load a lightweight embedding model (runs locally, no API cost)
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
message_1 = "Placed an order for some headphones early Monday morning, and they've arrived in 2 days... impressive service!"
message_2 = "Ordered this around 2am Friday morning and it made it here already... good job @115830 https://t.co/XXMuII3QwQ"
message_3 = "I hate Amazon!!!"

In [12]:
def get_embedding(text):
    """Get the embedding of an input text using a local model"""
    return embed_model.encode(text)

In [13]:
embedding_message_1 = get_embedding(message_1)
embedding_message_2 = get_embedding(message_2)
embedding_message_3 = get_embedding(message_3)

In [14]:
print(f"Message 1 : {message_1}\nEmbedding length : {len(embedding_message_1)}\nEmbedding (first 10 values):\n{embedding_message_1[:10]}")

Message 1 : Placed an order for some headphones early Monday morning, and they've arrived in 2 days... impressive service!
Embedding length : 384
Embedding (first 10 values):
[-0.06422088 -0.01477101  0.06188731 -0.03119966 -0.0779828  -0.03976501
 -0.00648614 -0.07367861 -0.05105711  0.01914502]


#### Compare two embeddings to find similarity

A cosine similarity close to 1 implies that the sentence embeddings are very similar, meaning their vectors point in almost the same direction. This suggests the sentences have similar meanings or semantic content.

A cosine similarity around 0 indicates that the sentence embeddings are orthogonal (or near-orthogonal) to each other in the vector space, suggesting that the sentences are unrelated or have neutral similarity.

A cosine similarity close to -1 indicates that the embeddings are diametrically opposed in the vector space, suggesting that the sentences are highly dissimilar or have opposite meanings.

In [15]:
def cosine_similarity(A, B):
    dot_product = np.dot(A, B)
    magnitude_A = np.linalg.norm(A)
    magnitude_B = np.linalg.norm(B)
    return dot_product / (magnitude_A * magnitude_B)

In [16]:
similarity_message1_message2 = cosine_similarity(embedding_message_1, embedding_message_2)
print(f"Message 1 : {message_1}\nMessage 2 : {message_2}\n\nSimilarity: {similarity_message1_message2}")

Message 1 : Placed an order for some headphones early Monday morning, and they've arrived in 2 days... impressive service!
Message 2 : Ordered this around 2am Friday morning and it made it here already... good job @115830 https://t.co/XXMuII3QwQ

Similarity: 0.459858238697052


In [17]:
similarity_message1_message3 = cosine_similarity(embedding_message_1, embedding_message_3)
print(f"Message 1 : {message_1}\nMessage 3 : {message_3}\n\nSimilarity: {similarity_message1_message3}")

Message 1 : Placed an order for some headphones early Monday morning, and they've arrived in 2 days... impressive service!
Message 3 : I hate Amazon!!!

Similarity: 0.29407942295074463


### Augment your prompt : RAG principle

In [18]:
# Customer's Tweet to answer 
tweet = "Placed an order for some headphones early Monday morning, and they've arrived in 2 days... impressive service!"

In [19]:
# Similar message with the agent's reply

similar_message = data_sample.head(1).customer_tweet[0]
reply = data_sample.head(1).company_tweet[0]
company = data_sample.head(1).company[0]

print(f"Customer's Tweet: {similar_message}\nCompany's Tweet: {reply}\nCompany: {company}")

Customer's Tweet: Ordered this around 2am Friday morning and it made it here already... good job @115830 https://t.co/XXMuII3QwQ
Company's Tweet: @383517 I am very happy to hear this Pablo:) I hope you enjoy your order.^GA
Company: AmazonHelp


In [20]:
instruction = f"""\
You are a chatbot answering customer's tweet. You are working for a company called Amazon. 
You are provided with an example of a similar interaction between a customer and an agent. Reply to the customer's tweet in the same tone, structure and style than the provided example.

#####
Example :
Customer : \"{similar_message}\"
Agent : \"{reply}\"

#####
Tweet:
\"{tweet}\"
"""
print(f"Instruction:\n\n{instruction}")

Instruction:

You are a chatbot answering customer's tweet. You are working for a company called Amazon. 
You are provided with an example of a similar interaction between a customer and an agent. Reply to the customer's tweet in the same tone, structure and style than the provided example.

#####
Example :
Customer : "Ordered this around 2am Friday morning and it made it here already... good job @115830 https://t.co/XXMuII3QwQ"
Agent : "@383517 I am very happy to hear this Pablo:) I hope you enjoy your order.^GA"

#####
Tweet:
"Placed an order for some headphones early Monday morning, and they've arrived in 2 days... impressive service!"



In [21]:
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": instruction}
    ],
    temperature=0.7
)

In [22]:
generated_text = response.content[0].text
print(f"Answer generated with RAG:\n\n{generated_text}")

Answer generated with RAG:

@[customer handle] That's wonderful to hear! We're thrilled you received your headphones so quickly:) We hope you love them.^GA


In [23]:
# Vanilla answer without augmentation

print(f"Vanilla answer without augmentation:\n\n{generated_text_vanilla}")

Vanilla answer without augmentation:

Thank you so much for the kind words! We're thrilled to hear that your headphones arrived so quickly and that you're impressed with our service. 🎉

Fast and reliable delivery is something we work hard to provide, so it's wonderful to know we met your expectations. We hope you're enjoying your new headphones!

If you have any questions about your order or need any assistance in the future, please don't hesitate to reach out. We're always here to help.

Thank you for shopping with Amazon! 😊


### Bonus: Using Claude with Tool Use (Function Calling)

One of Claude's powerful features is **tool use** — the ability to define functions that Claude can call. This is the foundation for building AI Agents (Project 2).

In [24]:
# Define a tool that Claude can use
tools = [
    {
        "name": "get_customer_info",
        "description": "Look up customer information by their username or ID. Returns customer name, account status, and recent orders.",
        "input_schema": {
            "type": "object",
            "properties": {
                "username": {
                    "type": "string",
                    "description": "The customer's username or Twitter handle"
                }
            },
            "required": ["username"]
        }
    }
]

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    tools=tools,
    messages=[
        {"role": "user", "content": "Can you look up the customer info for @john_doe123? They seem to have an issue with their order."}
    ]
)

print("Response stop reason:", response.stop_reason)
for block in response.content:
    print(f"Block type: {block.type}")
    if block.type == "tool_use":
        print(f"  Tool: {block.name}")
        print(f"  Input: {block.input}")
    elif block.type == "text":
        print(f"  Text: {block.text}")

Response stop reason: tool_use
Block type: tool_use
  Tool: get_customer_info
  Input: {'username': '@john_doe123'}
